In [1]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv(
    r"C:\Users\farrkhan\Documents\GitHub\customer-personality-analysis\data\raw\marketing_campaign.csv",
    sep="\t"
)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (2240, 29)


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


In [7]:
# 1. Check data types and non-null values
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   object 
 3   Marital_Status       2240 non-null   object 
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   object 
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases    2240 non-null   i

In [23]:
# 2. Check missing values
print("\n=== MISSING VALUES ===")

missing = df.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": (missing / len(df) * 100).round(2)
})

display(
    missing_summary[missing_summary["Missing_Count"] > 0]
    .sort_values("Missing_Count", ascending=False)
)


=== MISSING VALUES ===


,Missing_Count,Missing_Percentage
Income,24,1.07


In [27]:
3.# Check exact duplicate rows
print("Exact duplicate rows:", df.duplicated().sum())

# Check duplicate customer IDs
print("Duplicate customer IDs:", df["ID"].duplicated().sum())

# Count unique customer IDs
print("Unique customer IDs:", df["ID"].nunique())

# Check total rows
print("Total rows:", len(df))

Exact duplicate rows: 0
Duplicate customer IDs: 0
Unique customer IDs: 2240
Total rows: 2240


In [15]:
# 4. Check numeric statistics
print("\n=== NUMERIC SUMMARY ===")
display(df.describe().T)


=== NUMERIC SUMMARY ===


,count,mean,std,min,25%,50%,75%,max
ID,2240.0,5592.159821,3246.662198,0.0,2828.25,5458.5,8427.75,11191.0
Year_Birth,2240.0,1968.805804,11.984069,1893.0,1959.00,1970.0,1977.00,1996.0
Income,2216.0,52247.251354,25173.076661,1730.0,35303.00,51381.5,68522.00,666666.0
Kidhome,2240.0,0.444196,0.538398,0.0,0.00,0.0,1.00,2.0
Teenhome,2240.0,0.506250,0.544538,0.0,0.00,0.0,1.00,2.0
Recency,2240.0,49.109375,28.962453,0.0,24.00,49.0,74.00,99.0
MntWines,2240.0,303.935714,336.597393,0.0,23.75,173.5,504.25,1493.0
MntFruits,2240.0,26.302232,39.773434,0.0,1.00,8.0,33.00,199.0
MntMeatProducts,2240.0,166.950000,225.715373,0.0,16.00,67.0,232.00,1725.0
MntFishProducts,2240.0,37.525446,54.628979,0.0,3.00,12.0,50.00,259.0


In [37]:
#unusual birth years
display(
    df.loc[df["Year_Birth"] < 1900, ["ID", "Year_Birth"]]
)


,ID,Year_Birth
239,11004,1893
339,1150,1899


In [39]:
#unusually high income
display(
    df.loc[df["Income"] > 200000, ["ID", "Income"]]
)


,ID,Income
2233,9432,666666.0


In [19]:
# 5. Check categorical values
print("\n=== CATEGORICAL SUMMARY ===")

categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(10))


=== CATEGORICAL SUMMARY ===

Education:
Education
Graduation    1127
PhD            486
Master         370
2n Cycle       203
Basic           54
Name: count, dtype: int64

Marital_Status:
Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64

Dt_Customer:
Dt_Customer
31-08-2012    12
12-09-2012    11
14-02-2013    11
12-05-2014    11
20-08-2013    10
22-05-2014    10
05-04-2014     9
23-03-2014     9
02-01-2013     9
01-03-2014     9
Name: count, dtype: int64


In [43]:
#check customers date
customer_dates = pd.to_datetime(
    df["Dt_Customer"],
    format="%d-%m-%Y",
    errors="coerce"
)

print("\n=== CUSTOMER DATE CHECK ===")
print("Invalid/missing dates:", customer_dates.isna().sum())
print("Earliest date:", customer_dates.min())
print("Latest date:", customer_dates.max())


=== CUSTOMER DATE CHECK ===
Invalid/missing dates: 0
Earliest date: 2012-07-30 00:00:00
Latest date: 2014-06-29 00:00:00


In [47]:
display(
    df.loc[
        df["Year_Birth"] < 1900,
        ["ID", "Year_Birth", "Education",
         "Marital_Status", "Income"]
    ]
)


,ID,Year_Birth,Education,Marital_Status,Income
239,11004,1893,2n Cycle,Single,60182.0
339,1150,1899,PhD,Together,83532.0


In [50]:
display(
    df.loc[
        df["Income"] > 200000,
        ["ID", "Year_Birth", "Education",
         "Marital_Status", "Income"]
    ]
)

,ID,Year_Birth,Education,Marital_Status,Income
2233,9432,1977,Graduation,Together,666666.0


In [52]:
display(
    df.loc[
        df["Marital_Status"].isin(
            ["Alone", "Absurd", "YOLO"]
        ),
        ["ID", "Marital_Status", "Education",
         "Income", "Response"]
    ]
)


,ID,Marital_Status,Education,Income,Response
131,433,Alone,Master,61331.0,0
138,7660,Alone,PhD,35860.0,1
153,92,Alone,Graduation,34176.0,0
2093,7734,Absurd,Graduation,79244.0,1
2134,4369,Absurd,Master,65487.0,0
2177,492,YOLO,PhD,48432.0,0
2202,11133,YOLO,PhD,48432.0,1


In [54]:
display(
    df.loc[
        df["Income"].isna(),
        ["ID", "Year_Birth", "Education",
         "Marital_Status", "Kidhome", "Teenhome"]
    ]
)


,ID,Year_Birth,Education,Marital_Status,Kidhome,Teenhome
10,1994,1983,Graduation,Married,1,0
27,5255,1986,Graduation,Single,1,0
43,7281,1959,PhD,Single,0,0
48,7244,1951,Graduation,Single,2,1
58,8557,1982,Graduation,Single,1,0
71,10629,1973,2n Cycle,Married,1,0
90,8996,1957,PhD,Married,2,1
91,9235,1957,Graduation,Single,1,1
92,5798,1973,Master,Together,0,0
128,8268,1961,PhD,Married,0,1


In [72]:
print("Missing Income in original df:",
      df["Income"].isna().sum())

Missing Income in original df: 24


In [74]:
# Reset working copy
df_clean = df.copy()

# Mark missing income
df_clean["Income_Was_Missing"] = df_clean["Income"].isna()

# Calculate median
income_median = df_clean["Income"].median()

# Fill missing values
df_clean["Income"] = df_clean["Income"].fillna(income_median)

# Validate
print("Median used:", income_median)
print("Missing after cleaning:",
      df_clean["Income"].isna().sum())
print("Rows imputed:",
      df_clean["Income_Was_Missing"].sum())
print("Dataset shape:", df_clean.shape)

Median used: 51381.5
Missing after cleaning: 0
Rows imputed: 24
Dataset shape: (2240, 30)


In [76]:

# 11. Detect potential income outliers using IQR

# Use original income data and exclude missing values
income_valid = df["Income"].dropna()

# Calculate quartiles
q1 = income_valid.quantile(0.25)
q3 = income_valid.quantile(0.75)

# Calculate IQR and upper bound
iqr = q3 - q1
upper_bound = q3 + (1.5 * iqr)

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Upper bound:", upper_bound)

# Display potential outliers
income_outliers = df.loc[
    df["Income"] > upper_bound,
    ["ID", "Income"]
].sort_values("Income", ascending=False)

display(income_outliers)


Q1: 35303.0
Q3: 68522.0
IQR: 33219.0
Upper bound: 118350.5


,ID,Income
2233,9432,666666.0
617,1503,162397.0
687,1501,160803.0
1300,5336,157733.0
164,8475,157243.0
1653,4931,157146.0
2132,11181,156924.0
655,5555,153924.0


In [78]:
#investigate potential income outliers

outlier_cols = [
    "ID",
    "Year_Birth",
    "Education",
    "Marital_Status",
    "Income",
    "MntWines",
    "MntMeatProducts",
    "NumWebPurchases",
    "NumStorePurchases",
    "Response"
]

income_outliers_detail = df.loc[
    df["Income"] > upper_bound,
    outlier_cols
].sort_values("Income", ascending=False)

print("Number of potential outliers:", len(income_outliers_detail))

display(income_outliers_detail)


Number of potential outliers: 8


,ID,Year_Birth,Education,Marital_Status,Income,MntWines,MntMeatProducts,NumWebPurchases,NumStorePurchases,Response
2233,9432,1977,Graduation,Together,666666.0,9,18,3,3,0
617,1503,1976,PhD,Together,162397.0,85,16,0,1,0
687,1501,1982,PhD,Married,160803.0,55,1622,0,1,0
1300,5336,1971,Master,Together,157733.0,39,9,1,1,0
164,8475,1973,PhD,Married,157243.0,20,1582,0,0,0
1653,4931,1977,Graduation,Together,157146.0,1,1725,0,0,0
2132,11181,1949,PhD,Married,156924.0,2,2,0,0,0
655,5555,1975,Graduation,Divorced,153924.0,1,1,0,0,0


In [80]:

# flag potential income outliers

# Flag statistical outliers using the IQR upper bound
df_clean["Income_IQR_Outlier"] = (
    df_clean["Income"] > upper_bound
)

# Flag the extremely unusual income for further investigation
df_clean["Income_Extreme_Flag"] = (
    df_clean["Income"] == 666666
)

print(
    "IQR outliers:",
    df_clean["Income_IQR_Outlier"].sum()
)

print(
    "Extreme income records:",
    df_clean["Income_Extreme_Flag"].sum()
)

display(
    df_clean.loc[
        df_clean["Income_IQR_Outlier"],
        [
            "ID",
            "Income",
            "Income_IQR_Outlier",
            "Income_Extreme_Flag"
        ]
    ].sort_values("Income", ascending=False)
)


IQR outliers: 8
Extreme income records: 1


,ID,Income,Income_IQR_Outlier,Income_Extreme_Flag
2233,9432,666666.0,True,True
617,1503,162397.0,True,False
687,1501,160803.0,True,False
1300,5336,157733.0,True,False
164,8475,157243.0,True,False
1653,4931,157146.0,True,False
2132,11181,156924.0,True,False
655,5555,153924.0,True,False


In [82]:

#investigate the extreme income record

display(
    df_clean.loc[
        df_clean["Income_Extreme_Flag"],
        [
            "ID",
            "Year_Birth",
            "Education",
            "Marital_Status",
            "Income",
            "Kidhome",
            "Teenhome",
            "MntWines",
            "MntMeatProducts",
            "MntGoldProds",
            "NumWebPurchases",
            "NumStorePurchases",
            "NumCatalogPurchases",
            "Response"
        ]
    ]
)


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,MntWines,MntMeatProducts,MntGoldProds,NumWebPurchases,NumStorePurchases,NumCatalogPurchases,Response
2233,9432,1977,Graduation,Together,666666.0,1,0,9,18,12,3,3,1,0


In [84]:

#compariong income statistics with and without the extreme record

income_all = df_clean["Income"]

income_without_extreme = df_clean.loc[
    ~df_clean["Income_Extreme_Flag"],
    "Income"
]

print("=== ALL CUSTOMERS ===")
print("Mean income:", income_all.mean())
print("Median income:", income_all.median())

print("\n=== WITHOUT EXTREME RECORD ===")
print("Mean income:", income_without_extreme.mean())
print("Median income:", income_without_extreme.median())


=== ALL CUSTOMERS ===
Mean income: 52237.97544642857
Median income: 51381.5

=== WITHOUT EXTREME RECORD ===
Mean income: 51963.55471192497
Median income: 51381.5


In [86]:

#Flag unusual birth years

df_clean["Birth_Year_Flag"] = (
    df_clean["Year_Birth"] < 1900
)

print(
    "Potential birth-year anomalies:",
    df_clean["Birth_Year_Flag"].sum()
)

display(
    df_clean.loc[
        df_clean["Birth_Year_Flag"],
        [
            "ID",
            "Year_Birth",
            "Education",
            "Marital_Status",
            "Income"
        ]
    ]
)


Potential birth-year anomalies: 2


,ID,Year_Birth,Education,Marital_Status,Income
239,11004,1893,2n Cycle,Single,60182.0
339,1150,1899,PhD,Together,83532.0


In [88]:

# 17. Flag unusual marital status categories

unusual_status = ["Alone", "Absurd", "YOLO"]

df_clean["Marital_Status_Review"] = (
    df_clean["Marital_Status"].isin(unusual_status)
)

print(
    "Rows to review:",
    df_clean["Marital_Status_Review"].sum()
)

display(
    df_clean.loc[
        df_clean["Marital_Status_Review"],
        [
            "ID",
            "Marital_Status",
            "Education",
            "Year_Birth",
            "Income",
            "Response"
        ]
    ].sort_values("Marital_Status")
)


Rows to review: 7


,ID,Marital_Status,Education,Year_Birth,Income,Response
2093,7734,Absurd,Graduation,1993,79244.0,1
2134,4369,Absurd,Master,1957,65487.0,0
131,433,Alone,Master,1958,61331.0,0
138,7660,Alone,PhD,1973,35860.0,1
153,92,Alone,Graduation,1988,34176.0,0
2177,492,YOLO,PhD,1973,48432.0,0
2202,11133,YOLO,PhD,1973,48432.0,1


In [90]:

#standardize unusual marital status categories

df_clean["Marital_Status_Standardized"] = (
    df_clean["Marital_Status"].replace({
        "Absurd": "Other/Unclear",
        "YOLO": "Other/Unclear"
    })
)

print("=== STANDARDIZED MARITAL STATUS ==")

print(
    df_clean["Marital_Status_Standardized"]
    .value_counts(dropna=False)
)


=== STANDARDIZED MARITAL STATUS ===
Marital_Status_Standardized
Married          864
Together         580
Single           480
Divorced         232
Widow             77
Other/Unclear      4
Alone              3
Name: count, dtype: int64


In [92]:

# 19. Final data quality check

print("=== FINAL DATA QUALITY CHECK ===")

print("Dataset shape:", df_clean.shape)

print(
    "Total missing values:",
    df_clean.isnull().sum().sum()
)

print(
    "Exact duplicate rows:",
    df_clean.duplicated().sum()
)

print(
    "Duplicate customer IDs:",
    df_clean["ID"].duplicated().sum()
)

# Show columns that still contain missing values
missing_check = df_clean.isnull().sum()

display(
    missing_check[missing_check > 0]
)


=== FINAL DATA QUALITY CHECK ===
Dataset shape: (2240, 35)
Total missing values: 0
Exact duplicate rows: 0
Duplicate customer IDs: 0


Series([], dtype: int64)

In [94]:

# convert customer date to datetime

df_clean["Dt_Customer"] = pd.to_datetime(
    df_clean["Dt_Customer"],
    format="%d-%m-%Y",
    errors="coerce"
)

print("Date column type:", df_clean["Dt_Customer"].dtype)

print(
    "Invalid dates:",
    df_clean["Dt_Customer"].isna().sum()
)

print("Earliest date:", df_clean["Dt_Customer"].min())
print("Latest date:", df_clean["Dt_Customer"].max())


Date column type: datetime64[ns]
Invalid dates: 0
Earliest date: 2012-07-30 00:00:00
Latest date: 2014-06-29 00:00:00


In [96]:

from pathlib import Path

processed_dir = Path(
    r"C:\Users\farrkhan\Documents\GitHub\customer-personality-analysis\data\processed"
)

processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "marketing_campaign_clean.csv"

df_clean.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset saved successfully!")
print("File location:", output_path)
print("Dataset shape:", df_clean.shape)


Dataset saved successfully!
File location: C:\Users\farrkhan\Documents\GitHub\customer-personality-analysis\data\processed\marketing_campaign_clean.csv
Dataset shape: (2240, 35)
